# 🧠 Sesión 4 · Predecir un número con un CSV

Hasta ahora teníamos **1 dato** de entrada. Ahora usaremos una **tabla** (CSV) con **varias columnas**.

**Objetivo:** predecir el **precio de una vivienda** sabiendo sus **metros** y sus **habitaciones**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split

## 1. Crear el CSV
Esta celda crea un archivo `viviendas.csv` con 200 viviendas inventadas.

💡 Si tienes tu propio CSV, súbelo a Colab y sáltate esta celda.

In [ ]:
np.random.seed(0)
metros = np.random.randint(40, 150, 200)
habitaciones = np.random.randint(1, 5, 200)
precio = 2 * metros + 10 * habitaciones + np.random.normal(0, 10, 200)   # en miles de €

tabla = pd.DataFrame({"metros": metros, "habitaciones": habitaciones, "precio": precio.round(1)})
tabla.to_csv("viviendas.csv", index=False)
print("Archivo creado ✅")

## 2. Leer el CSV

In [ ]:
datos = pd.read_csv("viviendas.csv")
datos.head()

## 3. Separar preguntas y respuestas
- **X** = lo que sabemos (metros y habitaciones)
- **y** = lo que queremos predecir (precio)

In [ ]:
X = datos[["metros", "habitaciones"]]
y = datos["precio"]

## 4. Separar datos para entrenar y para examinar
Como en scikit-learn:
- **80 %** para que la red **aprenda**
- **20 %** para hacerle un **examen** con viviendas que no ha visto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("Para aprender:", len(X_train), "viviendas")
print("Para el examen:", len(X_test), "viviendas")

## 5. Crear la red
Lo único nuevo: `shape=(2,)` porque ahora entran **2 datos** (metros y habitaciones).

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(2,)),                      # 2 datos de entrada
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1)                         # 1 respuesta: el precio
])
modelo.compile(loss="mean_squared_error",
               optimizer=keras.optimizers.Adam(0.01))

## 6. Entrenar

In [ ]:
historial = modelo.fit(X_train, y_train, epochs=200, verbose=0)

plt.plot(historial.history["loss"])
plt.xlabel("Repeticiones")
plt.ylabel("Error")
plt.show()

## 7. El examen
Comparamos el **precio real** con el **precio que dice la red**.

In [ ]:
prediccion = modelo.predict(X_test, verbose=0).flatten()

resultado = pd.DataFrame({"precio real": y_test.values, "precio red": prediccion.round(1)})
resultado.head(10)

¿Cuánto se equivoca **de media**? (en miles de €)

In [ ]:
error_medio = np.mean(np.abs(y_test.values - prediccion))
print("La red se equivoca de media en", round(error_medio, 1), "mil €")

Gráfica: si la red acierta, los puntos caen **sobre la línea roja**.

In [ ]:
plt.scatter(y_test, prediccion)
plt.plot([100, 350], [100, 350], color="red")
plt.xlabel("Precio real")
plt.ylabel("Precio de la red")
plt.show()

## 8. Usar la red
¿Cuánto vale un piso de **90 metros** y **3 habitaciones**?

In [ ]:
piso = pd.DataFrame({"metros": [90], "habitaciones": [3]})
print(modelo.predict(piso, verbose=0), "mil €")

## 📝 Resumen
1. Leer el CSV con `pd.read_csv`
2. Separar **X** (columnas que sabemos) e **y** (lo que queremos predecir)
3. Separar **train** (aprender) y **test** (examen)
4. Crear la red con `shape=(número de columnas,)`
5. Entrenar, examinar y usar